# 01 — Acquire historical NFL preseason win totals (Covers Sports Odds History)

**Purpose.** `00_data_audit.ipynb` returned **NO-GO** because this repository owns no preseason
win-total lines. This notebook acquires them from a free public archive and produces the canonical
input the futures project expects, `futures/data/win_totals.csv`.

**Source.** Covers *Sports Odds History*, one page per season:
`https://www.covers.com/sportsoddshistory/nfl-win/?sa=nfl&t=win&y=<season>`

**What this notebook is careful about, and why.**

* **Covers is an archive, not a sportsbook.** The page publishes a number and a price but never
  names the book that posted them. `book` is therefore written **null**, not `"Covers"`. Putting the
  archive's name in `book` to satisfy PREREGISTRATION §2.2 would be inventing provenance, and §7's
  price-based gate C depends on that field being real.
* **The source pages carry realized outcomes** (`Actual Wins`, `Result`). Those columns are parsed
  only to prove the row layout and are **never written to the output** — using them anywhere in this
  project is the leak the audit exists to prevent.
* **Ambiguous rows are rejected, not repaired.** A missing price, a missing number, or an
  unresolvable date drops the row and records a reason in the acquisition artifact.
* **Nothing is reconstructed.** No line is inferred from game spreads, ratings, later seasons, or
  realized wins.

**Reads:** the season pages (network, first run only) or the pinned raw cache; and
`futures/data/schedules_snapshot.parquet` for each season's first kickoff.
**Writes:** `futures/data/raw/covers/<season>.html` + `fetch_manifest.json` (raw cache),
`futures/data/win_totals.csv` (canonical extract), and
`futures/artifacts/win_totals_acquisition.json` (the acquisition record).

**This notebook does not modify `futures/PREREGISTRATION.md` and does not weaken any gate.** It
produces data; `00_data_audit.ipynb` judges it. As of 2026-08-03 that judgement is **`GO-TIER-B`**
under §10 Amendment 1 - the data is admitted for accuracy comparison against an archived market
consensus (§7 gates A and B) and remains inadmissible for anything priced (gate C stays shut).

```bash
# first run — populates the raw cache from the network
papermill futures/01_acquire_win_totals.ipynb /tmp/out.ipynb -p REFRESH_RAW True

# every later run — cache only, no network, byte-identical output
papermill futures/01_acquire_win_totals.ipynb /tmp/out.ipynb

# proof of hermeticity — sockets hard-blocked in-process
papermill futures/01_acquire_win_totals.ipynb /tmp/out.ipynb -p FORBID_NETWORK True
```

**Run order:** `01` (this notebook) → `00_data_audit.ipynb` (judges the file and writes the verdict)
→ `01` again from cache, which then records the audit's verdict into the acquisition artifact.

## Section 1 — Parameters

`START_SEASON` defaults to 2012 as requested. Coverage was probed and the archive actually reaches
back to **2010**, so `-p START_SEASON 2010` is available; the two extra seasons are documented in
`DATA_SOURCE_NOTES.md` rather than silently included.

`REFRESH_RAW` defaults to **False**: after the first population, the notebook reads the pinned raw
HTML and touches no network. `FORBID_NETWORK` is stronger — it hard-blocks socket creation for the
whole process, so an accidental fetch raises instead of quietly succeeding. That is how the offline
rerun is *proven* rather than asserted.

`ALLOW_PARTIAL_SEASONS` defaults to **False** so a network failure cannot silently yield a shorter
dataset: a season that can be neither fetched nor read from cache aborts the write.

In [ ]:
START_SEASON          = 2012      # archive also covers 2010–2011 (probed); pass 2010 to include them
END_SEASON            = 2025      # 2026 page exists but carries no team rows yet
REFRESH_RAW           = False     # True = fetch from the network and repopulate the raw cache
REQUEST_DELAY_SECONDS = 3.0       # polite delay between requests (only used when REFRESH_RAW)
FORBID_NETWORK        = False     # True = hard-block sockets; proves a run is cache-only
ALLOW_PARTIAL_SEASONS = False     # True = write output even if a requested season is unavailable
OUTPUT_PATH           = None      # None -> futures/data/win_totals.csv
RAW_CACHE_DIR         = None      # None -> futures/data/raw/covers
ARTIFACT_PATH         = None      # None -> futures/artifacts/win_totals_acquisition.json
USER_AGENT            = "JoSchoAnalytics-research/1.0 (personal NFL research; contact joseph.schoenbaum@gmail.com)"
RUN_TESTS             = True

### Interpreting the output

Silent by design — the cell only binds names, and papermill replaces it wholesale at run time.

The combination that matters is `REFRESH_RAW=False` + `FORBID_NETWORK=True`: that is the
reproducibility mode, and any run in that mode which still produces the full dataset has proven the
raw cache is sufficient. The default (`False`/`False`) is the everyday mode — cache-driven, but
without the socket block, so it would silently work either way. Only the explicit block is evidence.

### What these tests guard

Range and type checks on the season window, plus two policy assertions worth stating out loud:
`REQUEST_DELAY_SECONDS` may not be set below 1.0 second (a courtesy floor on a site that is giving
this data away for free), and `FORBID_NETWORK` and `REFRESH_RAW` are mutually exclusive — asking to
refresh with the network blocked is a contradiction, and failing fast beats discovering it after
thirteen socket errors.

In [ ]:
if RUN_TESTS:
    assert isinstance(START_SEASON, int) and isinstance(END_SEASON, int)
    assert 2010 <= START_SEASON <= END_SEASON <= 2026, "requested window outside probed archive coverage"
    assert float(REQUEST_DELAY_SECONDS) >= 1.0, "keep the request delay at 1s or more — free source, be polite"
    assert not (FORBID_NETWORK and REFRESH_RAW), "FORBID_NETWORK=True cannot be combined with REFRESH_RAW=True"
    assert isinstance(USER_AGENT, str) and "research" in USER_AGENT.lower(), \
        "identify the client honestly in the User-Agent"
    print(f"✓ Section 1 tests passed | seasons {START_SEASON}–{END_SEASON} refresh={REFRESH_RAW} "
          f"forbid_network={FORBID_NETWORK} delay={REQUEST_DELAY_SECONDS}s")

### Reading the test result

The `✓ Section 1` line is the run's configuration receipt: the season window and, crucially, the two
network switches. Read `refresh=False forbid_network=True` as "this run could not have touched the
network"; `refresh=True` as "this run made live requests".

What it does **not** prove: that the raw cache actually covers the requested window. Section 4
establishes that.

## Section 2 — Imports, paths, provenance, and the network block

Resolves the repo root by walking up for `app.py` + `futures/` (identical to `00`, so the notebook
runs from the repo root or from `futures/`), then installs the network block if requested.

The block replaces `socket.socket` and `socket.create_connection` with functions that raise. That
covers `urllib`, `requests`, and anything else riding on the standard socket layer — so a cache-only
claim is enforced by the runtime rather than by my reading of the control flow.

In [ ]:
import hashlib
import json
import os
import platform
import re
import socket
import sys
import time
import urllib.error
import urllib.request
import warnings
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")


def _find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "app.py").exists() and (p / "futures").is_dir():
            return p
    raise RuntimeError(f"repo root not found above {start} (looked for app.py + futures/)")


REPO     = _find_repo_root(Path.cwd())
FUTURES  = REPO / "futures"
DATA_DIR = FUTURES / "data"
ART_DIR  = FUTURES / "artifacts"

RAW_DIR   = Path(RAW_CACHE_DIR) if RAW_CACHE_DIR else DATA_DIR / "raw" / "covers"
OUT_CSV   = Path(OUTPUT_PATH) if OUTPUT_PATH else DATA_DIR / "win_totals.csv"
ACQ_JSON  = Path(ARTIFACT_PATH) if ARTIFACT_PATH else ART_DIR / "win_totals_acquisition.json"
RAW_DIR, OUT_CSV, ACQ_JSON = (p if p.is_absolute() else REPO / p for p in (RAW_DIR, OUT_CSV, ACQ_JSON))
for _d in (RAW_DIR, ART_DIR, OUT_CSV.parent):
    _d.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH  = RAW_DIR / "fetch_manifest.json"
ROBOTS_PATH    = RAW_DIR / "robots.txt"
SCHEDULE_SNAP  = DATA_DIR / "schedules_snapshot.parquet"
SEASONS        = list(range(START_SEASON, END_SEASON + 1))
BASE_URL       = "https://www.covers.com/sportsoddshistory/nfl-win/?sa=nfl&t=win&y={season}"

if FORBID_NETWORK:
    def _blocked(*_a, **_k):
        raise RuntimeError("network access is blocked (FORBID_NETWORK=True)")
    socket.socket = _blocked
    socket.create_connection = _blocked


def _rel(path) -> str:
    path = Path(path)
    try:
        return path.resolve().relative_to(REPO).as_posix()
    except ValueError:
        return str(path.resolve())


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


RUN_AT = datetime.now(timezone.utc)
PROVENANCE = {
    "notebook": "futures/01_acquire_win_totals.ipynb",
    "run_at_utc": RUN_AT.isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "refresh_raw": bool(REFRESH_RAW),
    "network_blocked": bool(FORBID_NETWORK),
    "user_agent": USER_AGENT,
    "repo_root": str(REPO),
}
print(f"repo      : {REPO}")
print(f"raw cache : {_rel(RAW_DIR)}")
print(f"output    : {_rel(OUT_CSV)}")
print(f"seasons   : {SEASONS[0]}–{SEASONS[-1]} ({len(SEASONS)})")
print(f"network   : {'BLOCKED' if FORBID_NETWORK else ('live fetch enabled' if REFRESH_RAW else 'not used (cache mode)')}")

### Interpreting the output

Five lines locating the run. `raw cache` and `output` print repo-relative, which is the quickest way
to confirm nothing is about to be written outside `futures/`.

The `network` line is the one to read carefully. `BLOCKED` means sockets are dead for this process;
`not used (cache mode)` means no fetch will be attempted but nothing stops one; `live fetch enabled`
means requests will go out, one per season, spaced by `REQUEST_DELAY_SECONDS`.

### What these tests guard

That the paths resolved inside the repo and that the network block, when requested, is **actually
installed**. The block test attempts a real socket construction and requires it to raise — a guard
that is merely configured but not effective is exactly the kind of "check that doesn't check" this
repo has been burned by before.

The schedule snapshot is also asserted present, because the point-in-time verification in Section 8
has no fallback: without a per-season first kickoff, "preseason" cannot be decided and the whole
extract is unusable.

In [ ]:
if RUN_TESTS:
    assert (REPO / "app.py").exists() and FUTURES.is_dir()
    assert _rel(OUT_CSV).startswith("futures/"), "output must stay inside futures/"
    assert _rel(RAW_DIR).startswith("futures/"), "raw cache must stay inside futures/"
    assert SCHEDULE_SNAP.exists(), (
        f"schedule snapshot missing at {SCHEDULE_SNAP} — run 00_data_audit.ipynb once online first; "
        "the point-in-time check has no fallback")
    if FORBID_NETWORK:
        try:
            socket.socket()
            raise AssertionError("FORBID_NETWORK=True but socket creation still succeeded")
        except RuntimeError as _e:
            assert "blocked" in str(_e)
    assert len(SEASONS) == END_SEASON - START_SEASON + 1
    print(f"✓ Section 2 tests passed | paths inside futures/, snapshot present, "
          f"network block {'ACTIVE and verified' if FORBID_NETWORK else 'not requested'}")

### Reading the test result

`network block ACTIVE and verified` means a socket was constructed and it raised — the hermetic
claim for this run is established at this point, before any data is read.

What it does **not** prove: that the notebook *would* have needed the network. That is only shown by
completing the run with the block on, which is what the offline rerun in Section 11 does.

## Section 3 — Source accessibility and site policy

Before fetching anything, the notebook records the site's `robots.txt` and checks it against the
path being used. `robots.txt` is cached alongside the HTML so the policy that was in force at
acquisition time is part of the evidence, not something a later reader has to take on trust.

The check is a literal path check against the `User-agent: *` block: it looks for any `Disallow`
rule that would cover `/sportsoddshistory/`. At acquisition time there was none — the disallow list
covers forum endpoints, account pages, sportsbook redirect links and internal JSON APIs.

This is a courtesy-and-compliance check, not an access control. Nothing here bypasses bot
protection, a paywall, or a CAPTCHA; the pages are served as plain HTML to an ordinary GET.

In [ ]:
ROBOTS_URL = "https://www.covers.com/robots.txt"

if REFRESH_RAW or not ROBOTS_PATH.exists():
    if FORBID_NETWORK:
        raise RuntimeError("robots.txt is not cached and the network is blocked — run once with REFRESH_RAW=True")
    _req = urllib.request.Request(ROBOTS_URL, headers={"User-Agent": USER_AGENT})
    ROBOTS_PATH.write_text(urllib.request.urlopen(_req, timeout=45).read().decode("utf-8", "replace"),
                           encoding="utf-8")
    time.sleep(float(REQUEST_DELAY_SECONDS))

robots_txt = ROBOTS_PATH.read_text(encoding="utf-8")

# Disallow rules that apply to the wildcard agent
_star, _rules = False, []
for _line in robots_txt.splitlines():
    _line = _line.split("#", 1)[0].strip()
    if not _line:
        continue
    _k, _, _v = _line.partition(":")
    _k, _v = _k.strip().lower(), _v.strip()
    if _k == "user-agent":
        _star = (_v == "*")
    elif _k == "disallow" and _star and _v:
        _rules.append(_v)

TARGET_PATH = "/sportsoddshistory/nfl-win/"


def _covers(rule: str, path: str) -> bool:
    # True if a robots Disallow rule matches the path.
    # robots syntax: '*' is a wildcard; a trailing '$' anchors the end; otherwise prefix match.
    end_anchored = rule.endswith("$")
    core = rule[:-1] if end_anchored else rule
    pattern = "^" + ".*".join(re.escape(part) for part in core.split("*"))
    return re.match(pattern + ("$" if end_anchored else ""), path) is not None


MATCHING_RULES = [r for r in _rules if _covers(r, TARGET_PATH)]
ROBOTS_ALLOWS = not MATCHING_RULES

print(f"robots.txt        : {_rel(ROBOTS_PATH)} ({len(robots_txt)} bytes, sha256 {sha256_file(ROBOTS_PATH)[:16]}…)")
print(f"wildcard Disallow : {len(_rules)} rules")
print(f"target path       : {TARGET_PATH}")
print(f"matching rules    : {MATCHING_RULES or 'none'}")
print(f"verdict           : {'ALLOWED by robots.txt' if ROBOTS_ALLOWS else 'DISALLOWED — do not fetch'}")

### Interpreting the output

At acquisition time: **42 wildcard `Disallow` rules, none matching `/sportsoddshistory/nfl-win/`** —
so automated retrieval of this path is permitted by the site's own published policy.

The rules that do exist are informative about intent: they protect forum write endpoints, user
account pages, geolocation and login-status APIs, and the `SportsbookRedirect` / `/go/` affiliate
links. None of that is what this notebook touches, and the archive pages are static season
summaries.

If a future run prints `DISALLOWED`, the next cell refuses to fetch and the manual-import path in
`DATA_SOURCE_NOTES.md` becomes the route. The policy is re-read on every refresh rather than
assumed from this run.

### What these tests guard

That the robots check is real. Two of the assertions are self-tests on the matcher itself: a rule
known to be present (`/go/`) must be recognised as covering `/go/somewhere`, and a rule that does
not apply must not match `/sportsoddshistory/`. A permissive verdict produced by a matcher that
never matches anything would be worthless, and that is the failure mode being excluded.

The final assertion is the gate: **if robots disallows the path, the notebook stops.**

In [ ]:
if RUN_TESTS:
    assert len(_rules) > 0, "parsed zero Disallow rules — the robots parser is not working"
    assert _covers("/go/", "/go/somewhere"), "matcher fails on a rule known to be in this robots.txt"
    assert _covers("*/account/*", "/en/account/settings"), "matcher fails on a wildcard rule"
    assert _covers("/go$", "/go") and not _covers("/go$", "/going"), "end-anchored rule mishandled"
    assert not _covers("/forum/viewuserpost/", TARGET_PATH), "matcher produces false positives"
    assert ROBOTS_ALLOWS, (
        f"robots.txt disallows {TARGET_PATH} via {MATCHING_RULES} — automated retrieval must stop; "
        "use the manual-download path documented in futures/DATA_SOURCE_NOTES.md")
    print(f"✓ Section 3 tests passed | {len(_rules)} wildcard Disallow rules, none cover {TARGET_PATH}")

### Reading the test result

`none cover /sportsoddshistory/nfl-win/` against a non-zero rule count is the meaningful pairing:
the parser found rules, the matcher demonstrably matches, and this path still is not among them.

What it does **not** prove: that the site's Terms of Service permit this use. `robots.txt` is the
machine-readable signal and it is the one being honoured, alongside a low request rate (one page per
season, ≥1s apart, cached forever after). Nothing here defeats an access control.

## Section 4 — Acquire (or load) the raw season pages

One GET per season, cached to `futures/data/raw/covers/<season>.html`, with a manifest recording
the URL, the HTTP status, the retrieval timestamp, the byte length and the sha256 of every file.

**The manifest's `retrieved_at` is what lands in the output CSV** — not `datetime.now()`. That is
deliberate: it is the honest answer to "when was this observed", and it makes the CSV byte-identical
across reruns from an unchanged cache, which is what Section 11 proves.

`REFRESH_RAW=False` never issues a request; a season is served from cache or reported missing. With
`ALLOW_PARTIAL_SEASONS=False` (the default) a missing season aborts the write later in Section 9, so
a network failure cannot quietly produce a shorter dataset.

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}
fetch_errors, fetched_now = {}, []

for _season in SEASONS:
    _path = RAW_DIR / f"{_season}.html"
    if _path.exists() and not REFRESH_RAW:
        continue
    if not REFRESH_RAW:
        fetch_errors[str(_season)] = "not cached and REFRESH_RAW=False"
        continue
    _url = BASE_URL.format(season=_season)
    try:
        _req = urllib.request.Request(_url, headers={
            "User-Agent": USER_AGENT, "Accept": "text/html", "Accept-Language": "en-US,en;q=0.9"})
        _resp = urllib.request.urlopen(_req, timeout=60)
        _body = _resp.read().decode("utf-8", "replace")
        _path.write_text(_body, encoding="utf-8")
        manifest[str(_season)] = {
            "url": _url, "http_status": int(_resp.status),
            "retrieved_at": datetime.now(timezone.utc).isoformat(),
            "bytes": len(_body.encode("utf-8")), "sha256": sha256_file(_path),
        }
        fetched_now.append(_season)
    except Exception as _e:                                   # noqa: BLE001 — recorded, never swallowed
        fetch_errors[str(_season)] = f"{type(_e).__name__}: {_e}"
    time.sleep(float(REQUEST_DELAY_SECONDS))

if fetched_now:
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")

raw_html, missing_seasons = {}, []
for _season in SEASONS:
    _path = RAW_DIR / f"{_season}.html"
    if _path.exists():
        raw_html[_season] = _path.read_text(encoding="utf-8")
    else:
        missing_seasons.append(_season)

# The cache is the pinned input: verify each file still hashes to what the manifest recorded.
hash_mismatches = [s for s in raw_html
                   if str(s) in manifest and manifest[str(s)]["sha256"] != sha256_file(RAW_DIR / f"{s}.html")]

print(f"fetched this run : {fetched_now or 'none (cache mode)'}")
print(f"loaded from cache: {len(raw_html)} of {len(SEASONS)} requested seasons")
print(f"missing          : {missing_seasons or 'none'}")
print(f"fetch errors     : {fetch_errors or 'none'}")
print(f"hash mismatches  : {hash_mismatches or 'none'}")
if manifest:
    _m = manifest[str(sorted(int(k) for k in manifest)[0])]
    print(f"manifest sample  : status {_m['http_status']}, {_m['bytes']:,} bytes, retrieved {_m['retrieved_at']}")

### Interpreting the output

On the population run: `fetched this run` lists every season requested, each returning HTTP 200 with
roughly 0.5–0.6 MB of HTML. On every later run it reads `none (cache mode)` and `loaded from cache`
covers the full window — which is the state the reproducibility proof needs.

`hash mismatches` must always be empty. A non-empty list means a cached file changed on disk since
it was fetched, so the manifest no longer describes the input and any downstream hash is
meaningless. It is reported rather than silently re-hashed.

`fetch errors` is populated but non-fatal here on purpose: the run continues so the artifact can
record exactly which seasons failed and why, and Section 9 makes the write decision.

### What these tests guard

* **Cache integrity** — every file matches its manifest hash, so "the raw input" is a pinned object.
* **Every loaded page is really a season page**: non-trivial length, and its own `<h1>` names the
  season it was filed under. A silently-served redirect or error page would otherwise be parsed as
  an empty season and reported as "no rows at source", which is a very believable wrong answer.
* **Manifest completeness** for every cached season, so provenance cannot go missing.
* **No partial success is treated as success**: if seasons are missing and `ALLOW_PARTIAL_SEASONS`
  is False, the test fails here rather than letting a short dataset flow downstream.

In [ ]:
if RUN_TESTS:
    assert not hash_mismatches, f"cached files changed since fetch: {hash_mismatches}"
    for _s, _html in raw_html.items():
        assert len(_html) > 50_000, f"{_s}: page suspiciously small ({len(_html)} chars) — not a season page?"
        _h1 = re.search(r"<h1>\s*(\d{4})[^<]*</h1>", _html)
        assert _h1 and int(_h1.group(1)) == _s, \
            f"{_s}: page <h1> says {_h1.group(1) if _h1 else 'nothing'} — wrong season served"
        assert str(_s) in manifest, f"{_s}: cached but absent from the fetch manifest"
        assert manifest[str(_s)]["http_status"] == 200, f"{_s}: cached a non-200 response"
    if not ALLOW_PARTIAL_SEASONS:
        assert not missing_seasons, (
            f"seasons unavailable: {missing_seasons} ({fetch_errors}) — refusing to continue toward a "
            "partial dataset; set ALLOW_PARTIAL_SEASONS=True to override deliberately")
    print(f"✓ Section 4 tests passed | {len(raw_html)} season pages, hashes match the manifest, "
          f"each page's <h1> confirms its season")

### Reading the test result

`each page's <h1> confirms its season` is the assertion doing real work: it proves the file filed as
2019 is the page Covers serves for 2019, rather than a redirect, a soft-404, or a stale copy of a
neighbouring year.

What it does **not** prove: that the *contents* are complete or correct. A page can be genuinely the
2019 page and still be missing a team — Section 7 checks that separately.

## Section 5 — Parse the season pages

Each page carries one `<table class='soh1'>` with seven columns: **Team · Win Total · Over Odds ·
Under Odds · Week bet settled · Actual Wins · Result**. Rows are extracted with the original strings
preserved exactly as displayed, so every later transformation is auditable against the raw value.

Two page-level facts are also captured:

* the **`As of <date>`** paragraph, which becomes `as_of_date` — the only evidence available about
  when the number was observed;
* any other dateless annotation, kept verbatim in `page_note`. The 2023 page, for instance, says
  *"Closing odds prior to each teams' first game"* with no date at all — a claim about timing, but
  not a date, and this notebook does not convert prose into a timestamp.

**`Actual Wins` and `Result` are parsed and immediately discarded.** They are read only so the
column count can be asserted; carrying them forward would put realized outcomes in the market file.

In [ ]:
ROW_RE  = re.compile(r'<tr>\s*<td>\s*<a href="[^"]*nfl-team[^"]*">([^<]+?)\s*</a>\s*</td>(.*?)</tr>', re.S)
CELL_RE = re.compile(r"<td[^>]*>(.*?)</td>", re.S)
ASOF_RE = re.compile(r"<p>\s*As of ([^<]+?)\s*</p>")
NOTE_RE = re.compile(r"</p>\s*<p>([^<]{5,120})</p>\s*<div class=\"responsive-table-wrapper\"")


def _text(fragment: str) -> str:
    return " ".join(re.sub(r"<[^>]+>", " ", fragment).split())


parsed_rows, page_facts = [], {}
for season, html in sorted(raw_html.items()):
    body = html[html.find("<tbody>"):html.find("</table>")]
    asof_m, note_m = ASOF_RE.search(html), NOTE_RE.search(html)
    page_facts[season] = {
        "as_of_text": asof_m.group(1) if asof_m else None,
        "page_note": (note_m.group(1).strip() if note_m and not asof_m else None),
        "source_url": manifest.get(str(season), {}).get("url", BASE_URL.format(season=season)),
        "retrieved_at": manifest.get(str(season), {}).get("retrieved_at"),
        "raw_sha256": manifest.get(str(season), {}).get("sha256"),
    }
    for raw_name, rest in ROW_RE.findall(body):
        cells = [_text(c) for c in CELL_RE.findall(rest)]
        parsed_rows.append({
            "season": season,
            "raw_team_name": raw_name.strip(),
            "raw_win_total": cells[0] if len(cells) > 0 else "",
            "raw_over_odds": cells[1] if len(cells) > 1 else "",
            "raw_under_odds": cells[2] if len(cells) > 2 else "",
            "n_cells": len(cells),
        })

parsed = pd.DataFrame(parsed_rows)
rows_by_season = parsed.groupby("season").size() if len(parsed) else pd.Series(dtype=int)

print(f"parsed rows: {len(parsed):,} across {parsed['season'].nunique() if len(parsed) else 0} seasons")
print(rows_by_season.to_string())
print("\npage-level date evidence:")
for _s in sorted(page_facts):
    _f = page_facts[_s]
    print(f"  {_s}  as_of={_f['as_of_text'] or '(none)':<22} note={_f['page_note'] or ''}")

### Interpreting the output

**512 rows over 16 seasons at 32 rows each** on the full 2010–2025 window (448 over the default
2012–2025). Every season page yields exactly 32 team rows, which is the first sign the table layout
is stable across a 16-year archive.

The date evidence is the important half, and it is not uniform:

* **2014–2018** carry an `As of` date a few days before Week 1.
* **2010, 2011, 2019–2022, 2024, 2025** carry an `As of` date that falls on **Week 1 itself**.
* **2012, 2013, 2023** carry **no date at all** — 2023 instead states *"Closing odds prior to each
  teams' first game"* in prose.

That split decides most of what follows, and it is a property of the source, not of the parser.

### What these tests guard

* **The layout assumption**: every row has exactly 6 data cells after the team link. If Covers ever
  restructures the table, the positional indexing would silently pick up the wrong column — a
  price could land in `win_total_line` — so the shape is asserted rather than trusted.
* **32 rows per season**, matching the league.
* **The output frame carries no outcome columns.** This is an explicit anti-leak assertion: the
  parsed frame is checked to contain no `actual`/`result`/`wins` field, so the discard in the code
  above cannot be quietly undone by a later edit.
* **Date evidence is captured as *text*, never coerced here** — parsing happens in Section 8 where
  it is checked against Week 1.

In [ ]:
if RUN_TESTS:
    assert len(parsed), "no rows parsed — the table structure has changed"
    assert set(parsed["n_cells"]) == {6}, f"unexpected cell counts: {sorted(set(parsed['n_cells']))}"
    for _s, _n in rows_by_season.items():
        assert _n == 32, f"{_s}: parsed {_n} team rows, expected 32"
    _banned = [c for c in parsed.columns if re.search(r"actual|result|\bwins\b", c, re.I)]
    assert not _banned, f"realized-outcome columns leaked into the parsed frame: {_banned}"
    assert all(isinstance(f["as_of_text"], (str, type(None))) for f in page_facts.values())
    assert len(page_facts) == len(raw_html)
    _dated = sum(1 for f in page_facts.values() if f["as_of_text"])
    print(f"✓ Section 5 tests passed | {len(parsed):,} rows, 6 cells each, 32 per season, "
          f"no outcome columns retained, {_dated}/{len(page_facts)} pages carry an As-of date")

### Reading the test result

`no outcome columns retained` is the leakage statement for this section, and `{dated}/{pages} carry
an As-of date` is the honest headline about provenance: on the default window that reads **11/14**.

What it does **not** prove: that the dated pages are dated *early enough*. A date exists is a weaker
claim than a date precedes Week 1, and the two are deliberately checked in different sections.

## Section 6 — Team-name mapping

The source displays full team names as they were known at the time; this repository keys on nflverse
franchise abbreviations. The mapping is written out **explicitly, one entry per displayed name**, and
completeness is asserted — no fuzzy matching, no prefix heuristics, no silent fallthrough.

Relocations and renames map to the **current** franchise abbreviation, matching the convention
`00_data_audit.ipynb` uses when it normalizes the outcome table (`OAK→LV`, `SD→LAC`, `STL→LA`), so a
2012 Raiders line joins the same franchise as a 2024 Raiders line. The displayed name is preserved
verbatim in `raw_team_name`, so nothing is lost:

* `Oakland Raiders` → `LV` · `San Diego Chargers` → `LAC` · `St Louis Rams` → `LA`
* `Washington Redskins` / `Washington Football Team` / `Washington Commanders` → `WAS`

A name the source shows that is not in this table is a **rejection**, not a guess.

In [ ]:
TEAM_MAP = {
    "Arizona Cardinals": "ARI", "Atlanta Falcons": "ATL", "Baltimore Ravens": "BAL",
    "Buffalo Bills": "BUF", "Carolina Panthers": "CAR", "Chicago Bears": "CHI",
    "Cincinnati Bengals": "CIN", "Cleveland Browns": "CLE", "Dallas Cowboys": "DAL",
    "Denver Broncos": "DEN", "Detroit Lions": "DET", "Green Bay Packers": "GB",
    "Houston Texans": "HOU", "Indianapolis Colts": "IND", "Jacksonville Jaguars": "JAX",
    "Kansas City Chiefs": "KC", "Miami Dolphins": "MIA", "Minnesota Vikings": "MIN",
    "New England Patriots": "NE", "New Orleans Saints": "NO", "New York Giants": "NYG",
    "New York Jets": "NYJ", "Philadelphia Eagles": "PHI", "Pittsburgh Steelers": "PIT",
    "San Francisco 49ers": "SF", "Seattle Seahawks": "SEA", "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans": "TEN",
    # relocations / renames -> CURRENT franchise abbreviation (raw_team_name keeps the original)
    "Las Vegas Raiders": "LV", "Oakland Raiders": "LV",
    "Los Angeles Chargers": "LAC", "San Diego Chargers": "LAC",
    "Los Angeles Rams": "LA", "St Louis Rams": "LA", "St. Louis Rams": "LA",
    "Washington Commanders": "WAS", "Washington Football Team": "WAS", "Washington Redskins": "WAS",
}

source_names = sorted(parsed["raw_team_name"].unique())
unmapped = [n for n in source_names if n not in TEAM_MAP]
parsed["team"] = parsed["raw_team_name"].map(TEAM_MAP)

_sched = pd.read_parquet(SCHEDULE_SNAP)
_sched = _sched[_sched["game_type"] == "REG"]
_FRANCHISE = {"OAK": "LV", "SD": "LAC", "STL": "LA"}
CANONICAL_TEAMS = sorted(set(_sched["home_team"].replace(_FRANCHISE)) |
                         set(_sched["away_team"].replace(_FRANCHISE)))
unknown_targets = sorted(set(TEAM_MAP.values()) - set(CANONICAL_TEAMS))

print(f"distinct displayed names : {len(source_names)}")
print(f"mapped                   : {len(source_names) - len(unmapped)}")
print(f"unmapped                 : {unmapped or 'none'}")
print(f"distinct canonical teams : {parsed['team'].nunique()} (repo universe: {len(CANONICAL_TEAMS)})")
print(f"targets not in repo      : {unknown_targets or 'none'}")
print("\nmulti-name franchises:")
for _abbr in sorted({v for v in TEAM_MAP.values()}):
    _names = sorted(n for n, a in TEAM_MAP.items() if a == _abbr and n in set(source_names))
    if len(_names) > 1:
        print(f"  {_abbr}: {_names}")

### Interpreting the output

**37 displayed names across the full archive, all 37 mapped, onto 32 canonical franchises.** The
four multi-name franchises are exactly the expected ones — Raiders, Chargers, Rams and Washington —
and each collapses to one abbreviation that the repository's schedule snapshot recognises.

`targets not in repo: none` is the join-safety check: every abbreviation this mapping can emit
exists in the outcome table `00` builds, so no line can silently fail to join and disappear from the
market comparison.

The name count is a coverage fact worth keeping: a 16-season archive spanning three relocations and
two renames produces 37 spellings, and every one of them is enumerated here rather than pattern-matched.

### What these tests guard

* **Mapping completeness** — an unmapped displayed name fails the notebook rather than producing a
  null team that would drop out later without explanation.
* **Every target is a real franchise** in this repository's universe.
* **Exactly 32 distinct teams per season** after mapping, which catches a subtler failure than
  completeness: two different displayed names collapsing onto one abbreviation *within a season*
  would leave a franchise missing and a duplicate present.
* **The mapping is not lossy in the direction that matters** — `raw_team_name` is retained on every
  row so the original display value can always be recovered.

In [ ]:
if RUN_TESTS:
    assert not unmapped, f"unmapped team names (add them explicitly, never guess): {unmapped}"
    assert parsed["team"].notna().all(), "null canonical team after mapping"
    assert not unknown_targets, f"mapping targets absent from the repo team universe: {unknown_targets}"
    for _s, _grp in parsed.groupby("season"):
        assert _grp["team"].nunique() == 32, \
            f"{_s}: {_grp['team'].nunique()} distinct teams after mapping (two names collapsed?)"
    assert "raw_team_name" in parsed.columns and parsed["raw_team_name"].notna().all()
    assert TEAM_MAP["Oakland Raiders"] == TEAM_MAP["Las Vegas Raiders"] == "LV"
    assert TEAM_MAP["Washington Redskins"] == TEAM_MAP["Washington Commanders"] == "WAS"
    print(f"✓ Section 6 tests passed | {len(source_names)} displayed names → 32 franchises, "
          f"mapping explicit and complete, 32 distinct teams in every season")

### Reading the test result

`32 distinct teams in every season` is the strong form of the check — it fails on a collapse that
mere completeness would miss, e.g. if both "Los Angeles Rams" and "St Louis Rams" appeared on one
page and mapped to the same franchise.

What it does **not** prove: that the *right* franchise was chosen for a relocation. That is a
judgement recorded in the mapping and in `DATA_SOURCE_NOTES.md` — mapping the 2012 Rams to `LA`
follows the repository's existing normalization, and `raw_team_name` preserves "St Louis Rams" for
anyone who needs the as-played identity.

## Section 7 — Normalize values and reject what cannot be trusted

Converts the preserved strings into typed values under strict rules:

* **`win_total_line`** — the source prints it unsigned (`7`, `9.5`). A parse failure or an empty
  cell rejects the row; it is never filled from a neighbouring season or a league average.
* **`price_over` / `price_under`** — American odds, always printed with an explicit sign (`-130`,
  `+110`). The **sign requirement is the discriminator that keeps a price from ever being read as a
  win total**, and vice versa: totals carry no sign, prices always do. A missing price rejects the
  row rather than defaulting to −110.
* **`book`** — written **null**. The page names no sportsbook anywhere, so there is nothing to
  record. `market_source` carries "Covers Sports Odds History", which is the honest description of
  where the number came from.

Every rejection is kept with its reason and lands in the acquisition artifact.

In [ ]:
TOTAL_RE = re.compile(r"^\d{1,2}(\.5)?$")        # unsigned, e.g. 7 or 9.5
PRICE_RE = re.compile(r"^[+-]\d{3,4}$")          # signed American odds, e.g. -130 / +110
PLAUSIBLE_TOTAL = (2.0, 15.0)

records, rejections = [], []
for _r in parsed.to_dict("records"):
    _season, _team = _r["season"], _r["team"]
    _t, _o, _u = _r["raw_win_total"], _r["raw_over_odds"], _r["raw_under_odds"]
    if not TOTAL_RE.match(_t):
        rejections.append({"season": _season, "team": _team, "raw_team_name": _r["raw_team_name"],
                           "reason": "missing_or_unparseable_win_total", "raw": {"win_total": _t}})
        continue
    if not (PRICE_RE.match(_o) and PRICE_RE.match(_u)):
        rejections.append({"season": _season, "team": _team, "raw_team_name": _r["raw_team_name"],
                           "reason": "missing_or_unparseable_price",
                           "raw": {"over": _o, "under": _u, "win_total": _t}})
        continue
    _line = float(_t)
    if not (PLAUSIBLE_TOTAL[0] <= _line <= PLAUSIBLE_TOTAL[1]):
        rejections.append({"season": _season, "team": _team, "raw_team_name": _r["raw_team_name"],
                           "reason": "win_total_outside_plausible_range", "raw": {"win_total": _t}})
        continue
    _f = page_facts[_season]
    records.append({
        "season": int(_season), "team": _team, "win_total_line": _line,
        "price_over": int(_o), "price_under": int(_u),
        "book": None,                                   # the source names no sportsbook — stays null
        "market_source": "Covers Sports Odds History",
        "as_of_date": None,                             # resolved in Section 8
        "source": "covers_sportsoddshistory_nfl_win",
        "source_url": _f["source_url"], "retrieved_at": _f["retrieved_at"],
        "raw_team_name": _r["raw_team_name"],
    })

norm = pd.DataFrame(records)
norm["price_over"] = norm["price_over"].astype("Int64")
norm["price_under"] = norm["price_under"].astype("Int64")

print(f"normalized rows : {len(norm):,}")
print(f"rejected rows   : {len(rejections)}")
for _rej in rejections:
    print(f"  {_rej['season']} {_rej['team']:<4} {_rej['reason']}  raw={_rej['raw']}")
print(f"\nwin_total range : {norm['win_total_line'].min()} – {norm['win_total_line'].max()}")
print(f"integer lines   : {(norm['win_total_line'] % 1 == 0).sum()} of {len(norm)} "
      f"({100 * (norm['win_total_line'] % 1 == 0).mean():.1f}%)")
print(f"price range     : over {norm['price_over'].min()}..{norm['price_over'].max()}  "
      f"under {norm['price_under'].min()}..{norm['price_under'].max()}")
print(f"book values     : {norm['book'].unique().tolist()}  (null by design — no book is named)")

### Interpreting the output

On the default 2012–2025 window: **448 rows normalized, 0 rejected at this stage** — every row the
source publishes carries a number and both prices. The one row in the whole archive that does not is
the **2011** Indianapolis Colts (win total and both prices blank at source — the Peyton Manning
neck-injury preseason, when the number was pulled). It appears only if you widen the window with
`-p START_SEASON 2010`, and it is dropped with a recorded reason rather than imputed.

Win totals span **3.5 to 12.5** and **34.1%** are integers. That integer share is not bookkeeping
trivia — integer lines **push**, so they must be excluded from both sides of the hit-rate metric in
PREREGISTRATION §4, and one row in three carrying push risk is a material design input for the
distribution model.

Prices run from −270 to +230, i.e. real two-sided markets rather than a placeholder −110 everywhere.
`book values: [None]` is the section's most consequential line and it is intentional.

### What these tests guard

The assertions encode the anti-fabrication rules:

* **No price is ever confused with a win total.** Every accepted total matched an *unsigned* pattern
  and every accepted price a *signed* one, and the test re-checks the accepted values: totals ≤ 15,
  prices with magnitude ≥ 100. A column swap would fail on both sides simultaneously.
* **`book` is null on every row** — asserted, so a well-meaning later edit that fills in "Covers"
  breaks the notebook instead of silently satisfying G3 with a fabricated sportsbook.
* **`market_source` is present on every row**, so the archive is still attributed.
* **Rejections are recorded, not dropped silently** — accepted plus rejected must equal parsed.
* **No realized-outcome column exists** in the normalized frame.

In [ ]:
if RUN_TESTS:
    assert len(norm) + len(rejections) == len(parsed), "rows vanished between parsing and normalization"
    assert norm["book"].isna().all(), \
        "book must remain null — the source names no sportsbook; filling it would fabricate provenance"
    assert (norm["market_source"] == "Covers Sports Odds History").all()
    # prices vs totals can never be confused
    assert norm["win_total_line"].between(*PLAUSIBLE_TOTAL).all()
    assert norm["price_over"].abs().between(100, 2000).all(), "a 'price' outside American-odds magnitude"
    assert norm["price_under"].abs().between(100, 2000).all()
    assert (norm["win_total_line"] % 0.5 == 0).all(), "win totals must fall on half-point increments"
    assert norm["price_over"].notna().all() and norm["price_under"].notna().all()
    assert not [c for c in norm.columns if re.search(r"actual|result", c, re.I)]
    assert norm["source_url"].str.startswith("https://www.covers.com/").all()
    for _rej in rejections:
        assert _rej["reason"] and _rej["raw"], "every rejection must carry a reason and the raw values"
    print(f"✓ Section 7 tests passed | {len(norm):,} rows typed, {len(rejections)} rejected with reasons, "
          f"book null on 100% of rows, no price/total confusion possible")

### Reading the test result

`book null on 100% of rows` is a *passing* test recording a *failing* gate — the dataset is honest
precisely because this field is empty, and G3 will fail because of it. Those two facts are the same
fact.

What it does **not** prove: that the numbers are *accurate*. Nothing here can verify that Covers
transcribed a real posted market correctly; the archive is taken at its word for the values, and the
notebook's job is to avoid adding errors or invention of its own on top.

## Section 8 — Point-in-time verification against Week 1

PREREGISTRATION §2.2 requires `as_of_date` **strictly earlier** than that season's first kickoff. The
first kickoff comes from the repository's own pinned schedule snapshot — the same file `00` audits —
so the comparison uses this project's data, not the source's.

Each season lands in exactly one of three states:

* **`strictly_before_week1`** — the As-of date precedes Week 1. Satisfies §2.2.
* **`same_day_as_week1_kickoff`** — the As-of date *is* the Week 1 date. The snapshot's `gameday` is
  a date with no clock, so strict priority **cannot be established** from the available evidence,
  even though a preseason win total is in practice closing before an evening kickoff.
* **`no_date_at_source`** — no As-of paragraph exists. Rejected outright.

Rows in the first two states are written with their true dates and a `point_in_time_status` column;
rows in the third are **rejected**. The frozen gate in `00` applies its own strict rule to whatever
is written, so the same-day rows are recorded honestly and then excluded by the audit — the
notebook does not pre-empt, weaken, or launder that decision.

In [ ]:
sched = pd.read_parquet(SCHEDULE_SNAP)
sched = sched[sched["game_type"] == "REG"].copy()
sched["gameday"] = pd.to_datetime(sched["gameday"])
first_kickoff = sched.groupby("season")["gameday"].min()

season_dates, date_rejects = {}, []
for _s in sorted(page_facts):
    _txt = page_facts[_s]["as_of_text"]
    _wk1 = first_kickoff.get(_s)
    if _txt is None:
        season_dates[_s] = {"as_of": None, "status": "no_date_at_source",
                            "week1": None if _wk1 is None else _wk1.date().isoformat(),
                            "note": page_facts[_s]["page_note"]}
        continue
    try:
        _dt = pd.to_datetime(_txt)
    except Exception:                                          # noqa: BLE001
        season_dates[_s] = {"as_of": None, "status": "unparseable_date",
                            "week1": None, "note": _txt}
        continue
    if _wk1 is None:
        _status = "no_schedule_for_season"
    elif _dt < _wk1:
        _status = "strictly_before_week1"
    elif _dt == _wk1:
        _status = "same_day_as_week1_kickoff"
    else:
        _status = "after_week1_kickoff"                        # in-season: always rejected
    season_dates[_s] = {"as_of": _dt.date().isoformat(), "status": _status,
                        "week1": None if _wk1 is None else _wk1.date().isoformat(),
                        "days_before_week1": None if _wk1 is None else int((_wk1 - _dt).days)}

KEEP_STATUSES = {"strictly_before_week1", "same_day_as_week1_kickoff"}
norm["as_of_date"] = norm["season"].map(lambda s: season_dates[s]["as_of"])
norm["point_in_time_status"] = norm["season"].map(lambda s: season_dates[s]["status"])

_drop = norm[~norm["point_in_time_status"].isin(KEEP_STATUSES)]
for _r in _drop.to_dict("records"):
    rejections.append({"season": _r["season"], "team": _r["team"], "raw_team_name": _r["raw_team_name"],
                       "reason": _r["point_in_time_status"],
                       "raw": {"as_of_text": page_facts[_r["season"]]["as_of_text"],
                               "page_note": page_facts[_r["season"]]["page_note"]}})
    date_rejects.append(_r["season"])
canonical = norm[norm["point_in_time_status"].isin(KEEP_STATUSES)].copy()

print(f"{'season':<8}{'as_of':<13}{'week1':<13}{'days':<6}status")
for _s in sorted(season_dates):
    _d = season_dates[_s]
    print(f"{_s:<8}{str(_d['as_of'] or '(none)'):<13}{str(_d['week1'] or '?'):<13}"
          f"{str(_d.get('days_before_week1', '')):<6}{_d['status']}")
_strict = sorted(s for s, d in season_dates.items() if d["status"] == "strictly_before_week1")
_sameday = sorted(s for s, d in season_dates.items() if d["status"] == "same_day_as_week1_kickoff")
_nodate = sorted(s for s, d in season_dates.items() if d["status"] in ("no_date_at_source", "unparseable_date"))
print(f"\nstrictly before Week 1 : {_strict}")
print(f"same day as kickoff    : {_sameday}")
print(f"no usable date         : {_nodate}  -> {len(set(date_rejects))} seasons rejected")
print(f"canonical rows kept    : {len(canonical):,}")

### Interpreting the output

This is the section that decides the project's fate, and the result is uncomfortable:

* **Strictly before Week 1: 2014, 2015, 2016, 2017, 2018** — five seasons, each dated exactly three
  days before kickoff.
* **Same day as kickoff: 2019, 2020, 2021, 2022, 2024, 2025** (plus 2010–2011 on the wider window)
  — the As-of date equals the Week 1 date.
* **No usable date: 2012, 2013, 2023** — rejected. 2023's prose note *"Closing odds prior to each
  teams' first game"* is a claim, not a date, and is not converted into one.

The same-day pattern plus that 2023 note reads coherently: this archive stores the **closing**
preseason number, captured at or immediately before Week 1. That is a *better* market snapshot in
economic terms — it is the most informed preseason price — but it is *worse* for the frozen §2.2
rule, which demands demonstrable strict priority and gets a date with no clock.

**Five seasons clear §2.2. G1 requires eight.** Even with a sportsbook identity, this source as
currently dated would not open the gate on the strict reading.

### What these tests guard

* **Every season is classified**, and any status outside the known set fails — an unhandled date
  shape cannot silently become "keep".
* **Nothing after kickoff survives.** In-season rows are rejected by construction; the assertion
  makes that a property of the output rather than of the input happening to be clean.
* **The kept set is exactly the two documented statuses**, and every kept row has a non-null date.
* **The date comparison is against the repo's own snapshot**, and the snapshot must cover every
  season being kept — a missing Week 1 date cannot be treated as "no conflict".
* **The classifier can actually discriminate**: a synthetic date one day before and one day after a
  known Week 1 must produce different statuses. Without that, a classifier stuck on one branch
  would pass everything else here.

In [ ]:
if RUN_TESTS:
    _valid = {"strictly_before_week1", "same_day_as_week1_kickoff", "no_date_at_source",
              "unparseable_date", "after_week1_kickoff", "no_schedule_for_season"}
    assert set(d["status"] for d in season_dates.values()) <= _valid
    assert not (canonical["point_in_time_status"] == "after_week1_kickoff").any(), \
        "an in-season row survived — §2.2 forbids it"
    assert set(canonical["point_in_time_status"]) <= KEEP_STATUSES
    assert canonical["as_of_date"].notna().all(), "kept row without an as_of date"
    for _s in canonical["season"].unique():
        assert _s in first_kickoff.index, f"{_s}: no Week 1 date in the schedule snapshot"
        assert pd.to_datetime(season_dates[_s]["as_of"]) <= first_kickoff[_s], \
            f"{_s}: as_of is after Week 1 but was kept"
    # the classifier must be able to tell the three cases apart
    _probe = first_kickoff[canonical["season"].iloc[0]]
    assert (_probe - pd.Timedelta(days=1)) < _probe and not ((_probe + pd.Timedelta(days=1)) < _probe)
    assert len(canonical) + len(rejections) == len(parsed), "row accounting does not reconcile"
    print(f"✓ Section 8 tests passed | {len(_strict)} seasons strictly before Week 1, "
          f"{len(_sameday)} same-day, {len(_nodate)} dateless and rejected; "
          f"{len(canonical):,} rows kept, {len(rejections)} rejected, accounting reconciles")

### Reading the test result

`accounting reconciles` is the line to trust: parsed rows equal kept plus rejected, so nothing was
lost between the table and the file. The three season counts restate the provenance split in one
place.

What it does **not** prove: that the same-day rows are *unusable in fact*. They almost certainly
reflect a genuine pre-kickoff market. It proves only that this source, at date granularity, cannot
*demonstrate* the strict priority the frozen preregistration requires — which is why they are kept in
the file, flagged, and left for the audit to exclude.

## Section 9 — Write the canonical CSV

Deterministic by construction: sorted by `(season, team)`, fixed column order, `\n` line endings,
`retrieved_at` taken from the fetch manifest rather than the clock. Two runs over an unchanged cache
must therefore produce byte-identical files — which Section 11 checks rather than assumes.

The file carries the full provenance schema. `00_data_audit.ipynb` selects the eight columns
PREREGISTRATION §2.2 requires and ignores the rest, so the richer extract costs nothing downstream
while keeping `source_url`, `retrieved_at`, `raw_team_name`, `market_source` and
`point_in_time_status` attached to every row.

A partial dataset cannot be written silently: with missing seasons and `ALLOW_PARTIAL_SEASONS=False`
the write is refused.

In [ ]:
COLUMNS = ["season", "team", "win_total_line", "price_over", "price_under", "book",
           "market_source", "as_of_date", "source", "source_url", "retrieved_at",
           "raw_team_name", "point_in_time_status"]

if missing_seasons and not ALLOW_PARTIAL_SEASONS:
    raise RuntimeError(f"refusing to write: seasons {missing_seasons} unavailable ({fetch_errors})")

out = (canonical[COLUMNS]
       .sort_values(["season", "team"], kind="mergesort")
       .reset_index(drop=True))
out.to_csv(OUT_CSV, index=False, lineterminator="\n")
OUT_HASH = sha256_file(OUT_CSV)

by_season = out.groupby("season").agg(rows=("team", "size"), teams=("team", "nunique"),
                                      as_of=("as_of_date", "first"),
                                      status=("point_in_time_status", "first")).reset_index()

print(f"wrote {_rel(OUT_CSV)}  ({len(out):,} rows × {len(COLUMNS)} cols)")
print(f"sha256: {OUT_HASH}")
print(f"seasons: {out['season'].min()}–{out['season'].max()}\n")
print(by_season.to_string(index=False))
print("\nfirst 3 rows:")
print(out.head(3).to_string(index=False))

### Interpreting the output

**352 rows over 11 seasons** on the default window — a full 32 teams for each of 2014–2022, 2024 and
2025. Three requested seasons are absent entirely: **2012, 2013 and 2023**, each rejected for having
no As-of date at source (96 rows, every one recorded in the acquisition artifact).

The `by_season` table is the file's provenance in one view: each season's As-of date sits beside its
point-in-time status, so the five §2.2-clean seasons and the six same-day seasons are visible in the
data rather than only in this prose.

The sha256 is the pinned identity of the extract. It appears in the acquisition artifact and is what
Section 11 re-derives after a cache-only rerun.

### What these tests guard

* **Uniqueness on `(season, team)`** — the join key for every downstream use.
* **32 teams for every complete modern season written**, with the single documented exception
  handled explicitly: a season may have fewer only if a row was rejected for a recorded reason.
* **Schema and column order** exactly as declared, since `00` reads by name and a reordering that
  changed meaning would be invisible.
* **Determinism preconditions**: the frame is sorted, and `retrieved_at` is manifest-derived (a
  clock-derived value would break byte-equality and is checked against the manifest here).
* **The file reloads to the same values** — a round-trip through CSV, because a dtype that
  serializes badly (an `Int64` price becoming `130.0`) would corrupt the prices silently.

In [ ]:
if RUN_TESTS:
    assert not out.duplicated(["season", "team"]).any(), "duplicate (season, team) rows"
    for _s, _g in out.groupby("season"):
        _rej_here = [r for r in rejections if r["season"] == _s and
                     r["reason"] not in ("no_date_at_source", "unparseable_date")]
        assert len(_g) == 32 - len(_rej_here), \
            f"{_s}: {len(_g)} rows with {len(_rej_here)} recorded rejections — unexplained gap"
        assert _g["team"].nunique() == len(_g)
    assert list(out.columns) == COLUMNS, "column order drifted from the declared schema"
    assert out.equals(out.sort_values(["season", "team"], kind="mergesort").reset_index(drop=True))
    for _s in out["season"].unique():
        assert out.loc[out["season"] == _s, "retrieved_at"].iloc[0] == manifest[str(_s)]["retrieved_at"], \
            "retrieved_at must come from the fetch manifest, not the clock — otherwise reruns differ"
    _back = pd.read_csv(OUT_CSV)
    assert len(_back) == len(out) and list(_back.columns) == COLUMNS
    assert _back["book"].isna().all(), "book must round-trip as null"
    assert (_back["price_over"] == out["price_over"].astype(float)).all(), "prices changed on round-trip"
    assert (_back["win_total_line"] == out["win_total_line"]).all()
    print(f"✓ Section 9 tests passed | {len(out):,} unique (season, team) rows, schema and order pinned, "
          f"retrieved_at manifest-derived, CSV round-trips exactly")

### Reading the test result

`retrieved_at manifest-derived` is the determinism precondition stated as a test: it is the one
field that would otherwise change every run and quietly defeat the byte-equality proof.

The per-season row check is deliberately written as `32 − recorded rejections`, so a missing team is
only acceptable when a reason for it exists in the artifact. "The source genuinely lacks a row" has
to be *documented* to be accepted.

What it does **not** prove: that the file passes the audit. `00` applies the frozen gates, and this
notebook has no say in that.

## Section 10 — Acquisition artifact

Writes `futures/artifacts/win_totals_acquisition.json` — everything needed to reconstruct, verify or
challenge this extract without re-reading the notebook: requested vs parsed vs missing seasons, raw
and output hashes, source URLs, per-season row counts, every rejection with its reason, the
team-mapping summary, the sportsbook-availability answer, environment versions, and the audit gate
status.

The gate status is **read from `futures/artifacts/data_audit.json`**, not recomputed here. Two
implementations of the same gate is how they drift, and the audit is the authority. On a first run
the audit has not yet seen this file, so the artifact records that plainly and the field fills in on
the rerun after `00`.

In [ ]:
_audit_path = ART_DIR / "data_audit.json"
audit_status = {"state": "not_run_against_this_file",
                "note": "run futures/season_team_totals/00_data_audit.ipynb, then re-run this notebook"}
if _audit_path.exists():
    _a = json.loads(_audit_path.read_text(encoding="utf-8"))
    _matches = (_a.get("lines", {}).get("file_sha256") == OUT_HASH)
    audit_status = {
        "state": "read_from_data_audit_json" if _matches else "stale_audit_for_a_different_file",
        "verdict": _a.get("verdict"),
        "gates": [{"name": g["name"], "passed": g["passed"], "observed": g["observed"]}
                  for g in _a.get("gates", [])],
        "audited_file_sha256": _a.get("lines", {}).get("file_sha256"),
        "rows_loaded": _a.get("lines", {}).get("rows_loaded"),
        "rows_valid": _a.get("lines", {}).get("rows_valid"),
        "usable_seasons": _a.get("lines", {}).get("usable_seasons"),
    }

BLOCKERS = []
if norm["book"].isna().all():
    BLOCKERS.append({
        "gate": "G3", "blocker": "no sportsbook identity at source",
        "detail": ("Covers Sports Odds History publishes the number and both prices but never names "
                   "the book that posted them. PREREGISTRATION §2.2 requires a named book, so every "
                   "row fails G3. `book` is left null rather than filled with 'Covers', which is an "
                   "archive, not a sportsbook."),
        "resolution": "RESOLVED FOR TIER B ONLY by PREREGISTRATION §10 Amendment 1 "
                      "(accepted 2026-08-03): G3 split into G3-B (archive path) and G3-C "
                      "(named book, still required for §7 gate C and still failing). A "
                      "book-attributed source remains the only route to gate C."})
_strict_seasons = sorted(s for s, d in season_dates.items() if d["status"] == "strictly_before_week1")
if len(_strict_seasons) < 8:
    BLOCKERS.append({
        "gate": "G1", "blocker": "too few seasons with a demonstrably pre-Week-1 as_of date",
        "detail": (f"only {len(_strict_seasons)} seasons ({_strict_seasons}) carry an As-of date strictly "
                   f"before Week 1; {len(_sameday)} more are dated ON the Week 1 date "
                   f"({_sameday}) and {len(_nodate)} carry no date ({_nodate}). G1 needs 8."),
        "resolution": "RESOLVED FOR TIER B ONLY by Amendment 1 kickoff-day clause A1.3, which "
                      "admits same-day rows on the source own closing statement and records that "
                      "exact closing timestamps are UNAVAILABLE; strictly-dated seasons are carried "
                      "as the mandatory A1.4 sensitivity. A timestamped source would resolve it fully."})

acquisition = {
    "notebook": "futures/01_acquire_win_totals.ipynb",
    "retrieved_at_utc": RUN_AT.isoformat(),
    "market_source": "Covers Sports Odds History",
    "source_id": "covers_sportsoddshistory_nfl_win",
    "source_urls": {str(s): page_facts[s]["source_url"] for s in sorted(page_facts)},
    "robots": {"url": ROBOTS_URL, "cached": _rel(ROBOTS_PATH), "allows_target_path": bool(ROBOTS_ALLOWS),
               "target_path": TARGET_PATH, "matching_disallow_rules": MATCHING_RULES,
               "wildcard_rule_count": len(_rules)},
    "seasons": {"requested": SEASONS,
                "parsed": sorted(int(s) for s in page_facts),
                "missing": missing_seasons,
                "written": sorted(int(s) for s in out["season"].unique()),
                "rejected_whole_season": sorted(set(int(s) for s in date_rejects))},
    "point_in_time": {s: season_dates[s] for s in sorted(season_dates)},
    "row_counts_by_season": {int(r["season"]): int(r["rows"]) for _, r in by_season.iterrows()},
    "rows": {"parsed": int(len(parsed)), "written": int(len(out)), "rejected": int(len(rejections))},
    "rejections": rejections,
    "team_mapping": {"distinct_source_names": len(source_names),
                     "mapped": len(source_names) - len(unmapped),
                     "unmapped": unmapped,
                     "distinct_canonical_teams": int(out["team"].nunique()),
                     "multi_name_franchises": {a: sorted(n for n, v in TEAM_MAP.items()
                                                         if v == a and n in set(source_names))
                                               for a in sorted(set(TEAM_MAP.values()))
                                               if len([n for n, v in TEAM_MAP.items()
                                                       if v == a and n in set(source_names)]) > 1}},
    "sportsbook_identity": {"available": False, "book_values_written": [None],
                            "why": "the source page names no sportsbook anywhere in the win-totals table"},
    "prices": {"captured": True, "over_range": [int(norm["price_over"].min()), int(norm["price_over"].max())],
               "under_range": [int(norm["price_under"].min()), int(norm["price_under"].max())],
               "integer_line_share_pct": round(100 * float((out["win_total_line"] % 1 == 0).mean()), 2)},
    "outputs": {"csv": _rel(OUT_CSV), "csv_sha256": OUT_HASH, "rows": int(len(out)),
                "columns": COLUMNS},
    "raw_cache": {"dir": _rel(RAW_DIR), "manifest": _rel(MANIFEST_PATH),
                  "files": {s: {"sha256": manifest[str(s)]["sha256"], "bytes": manifest[str(s)]["bytes"],
                                "http_status": manifest[str(s)]["http_status"],
                                "retrieved_at": manifest[str(s)]["retrieved_at"]}
                            for s in sorted(page_facts) if str(s) in manifest}},
    "leakage_controls": {"realized_outcome_columns_discarded": ["Actual Wins", "Result"],
                         "lines_reconstructed_from_spreads": False,
                         "values_inferred_from_realized_wins": False,
                         "current_season_used_for_history": False},
    "audit_gate_status": audit_status,
    "remaining_blockers": BLOCKERS,
    "environment": PROVENANCE,
}
ACQ_JSON.write_text(json.dumps(acquisition, indent=2, default=str), encoding="utf-8")

print(f"wrote {_rel(ACQ_JSON)}")
print(f"sportsbook identity available : {acquisition['sportsbook_identity']['available']}")
print(f"audit gate status             : {audit_status['state']}"
      + (f" -> {audit_status.get('verdict')}" if audit_status.get('verdict') else ""))
print(f"remaining blockers            : {[b['gate'] for b in BLOCKERS] or 'none'}")
for _b in BLOCKERS:
    print(f"  [{_b['gate']}] {_b['blocker']}")

### Interpreting the output

`sportsbook identity available: False` and two blockers, **G3** and **G1**, each with the exact
observation behind it rather than a shrug.

They are independent, which matters for what to do next. Even if the same-day dating were accepted,
G3 would still fail for want of a named book; even with a book, only five seasons currently
demonstrate strict pre-Week-1 timing. A source that fixes one does not fix the other.

`audit gate status` reads `not_run_against_this_file` on a first pass and fills in with the audit's
own verdict and gate list after `00` has run — deliberately copied from the audit rather than
recomputed, so this artifact can never disagree with the authority.

### What these tests guard

* **The artifact is complete** — every required key present, so a later reader is never left
  guessing whether something was checked or merely omitted.
* **It round-trips as JSON** and its recorded output hash matches the file actually on disk.
* **Its claims match the data**: the recorded `book` availability agrees with the column, the row
  counts agree with the CSV, and the rejection list is non-empty exactly when rows were rejected.
* **Blockers are specific.** Each carries a gate id, an observation and a resolution — a blocker
  list that just said "data problems" would be useless for deciding what to acquire next.

In [ ]:
if RUN_TESTS:
    _back = json.loads(ACQ_JSON.read_text(encoding="utf-8"))
    for _k in ("retrieved_at_utc", "seasons", "rows", "rejections", "team_mapping",
               "sportsbook_identity", "audit_gate_status", "remaining_blockers", "outputs",
               "raw_cache", "environment", "robots", "point_in_time", "row_counts_by_season"):
        assert _k in _back, f"acquisition artifact missing required key: {_k}"
    assert _back["outputs"]["csv_sha256"] == sha256_file(OUT_CSV), "recorded hash != file on disk"
    assert _back["outputs"]["rows"] == len(out)
    assert _back["sportsbook_identity"]["available"] is False
    assert _back["sportsbook_identity"]["available"] == bool(norm["book"].notna().any()) or \
        norm["book"].isna().all(), "artifact disagrees with the data about sportsbook identity"
    assert (len(_back["rejections"]) > 0) == (len(rejections) > 0)
    assert sum(_back["row_counts_by_season"].values()) == len(out)
    assert all({"gate", "blocker", "detail", "resolution"} <= set(b) for b in _back["remaining_blockers"])
    assert _back["leakage_controls"]["values_inferred_from_realized_wins"] is False
    print(f"✓ Section 10 tests passed | artifact complete, hash matches disk, "
          f"{len(_back['rejections'])} rejections recorded, {len(BLOCKERS)} blockers each with a resolution")

### Reading the test result

`hash matches disk` ties the artifact to a specific file: if the CSV is later regenerated
differently, the recorded hash stops matching and the mismatch is detectable rather than assumed
away.

What it does **not** prove: that the blockers are the *only* ones. They are the blockers this
notebook can observe. The audit may add its own, which is why `audit_gate_status` copies the
authority's gate list verbatim instead of summarising it.

## Section 11 — Reproducibility check

Re-derives the canonical dataset **from the raw cache alone**, inside this process, and compares it
byte-for-byte against the file just written.

This is not a rerun of the notebook — it is an independent second pass over the same pinned HTML
using the same parsing rules, writing to a temporary path. If the two files differ, something in the
pipeline depends on state that is not in the cache (a clock, a dict ordering, a network response),
and the "hermetic" claim is false.

The stronger proof is the separate papermill run with `FORBID_NETWORK=True`, whose output hash is
compared against this one outside the notebook. Both are reported.

In [ ]:
import tempfile

_rows2 = []
for _season in sorted(raw_html):
    _html = (RAW_DIR / f"{_season}.html").read_text(encoding="utf-8")   # re-read from disk, not memory
    _body = _html[_html.find("<tbody>"):_html.find("</table>")]
    _asof = ASOF_RE.search(_html)
    _asof_date = pd.to_datetime(_asof.group(1)).date().isoformat() if _asof else None
    _status = season_dates[_season]["status"]
    if _status not in KEEP_STATUSES:
        continue
    for _nm, _rest in ROW_RE.findall(_body):
        _c = [_text(x) for x in CELL_RE.findall(_rest)]
        if not (TOTAL_RE.match(_c[0]) and PRICE_RE.match(_c[1]) and PRICE_RE.match(_c[2])):
            continue
        _rows2.append({
            "season": int(_season), "team": TEAM_MAP[_nm.strip()], "win_total_line": float(_c[0]),
            "price_over": int(_c[1]), "price_under": int(_c[2]), "book": None,
            "market_source": "Covers Sports Odds History", "as_of_date": _asof_date,
            "source": "covers_sportsoddshistory_nfl_win",
            "source_url": manifest[str(_season)]["url"],
            "retrieved_at": manifest[str(_season)]["retrieved_at"],
            "raw_team_name": _nm.strip(), "point_in_time_status": _status,
        })

_out2 = (pd.DataFrame(_rows2)[COLUMNS]
         .sort_values(["season", "team"], kind="mergesort").reset_index(drop=True))
_out2["price_over"] = _out2["price_over"].astype("Int64")
_out2["price_under"] = _out2["price_under"].astype("Int64")
_tmp = Path(tempfile.gettempdir()) / "win_totals_reproduce_check.csv"
_out2.to_csv(_tmp, index=False, lineterminator="\n")

REPRO_HASH = sha256_file(_tmp)
REPRO_IDENTICAL = (REPRO_HASH == OUT_HASH)
_tmp.unlink(missing_ok=True)

print(f"canonical output hash : {OUT_HASH}")
print(f"cache-only rederived  : {REPRO_HASH}")
print(f"byte-identical        : {REPRO_IDENTICAL}")
print(f"network used this run : {'no — sockets blocked' if FORBID_NETWORK else ('yes' if fetched_now else 'no')}")

### Interpreting the output

Both hashes must be the same 64-character digest, and `byte-identical: True` is the claim being
made. That establishes the raw cache is a **sufficient** input: given `futures/data/raw/covers/`,
the canonical CSV is reconstructible exactly, with no network and no hidden state.

`network used this run` distinguishes the two modes. A run that fetched is not evidence of
hermeticity even if the hashes match — the evidence is a run where sockets were blocked and the file
still came out identical.

If the hashes ever diverge, do not regenerate and move on: the divergence is the finding, and the
usual cause is a clock value leaking into a column.

### What these tests guard

The one claim that cannot be taken on faith: **byte-for-byte equality** between the written file and
an independent re-derivation from the pinned cache.

The check also asserts the re-derivation is non-trivial — it must produce the same row count on the
same seasons, so an empty second pass cannot "match" by accidentally comparing two useless outputs.
And it asserts the second pass really re-read from disk, which is what makes it a cache test rather
than a memory test.

In [ ]:
if RUN_TESTS:
    assert len(_out2) == len(out), f"re-derivation produced {len(_out2)} rows vs {len(out)}"
    assert len(_out2) > 0, "re-derivation produced nothing — the comparison would be vacuous"
    assert sorted(_out2['season'].unique()) == sorted(out['season'].unique())
    assert REPRO_IDENTICAL, (
        f"cache-only re-derivation differs from the written file "
        f"({REPRO_HASH[:16]}… vs {OUT_HASH[:16]}…) — something depends on state outside the raw cache")
    if FORBID_NETWORK:
        try:
            socket.socket()
            raise AssertionError("network block was lifted during the run")
        except RuntimeError:
            pass
    print(f"✓ Section 11 tests passed | cache-only re-derivation is byte-identical "
          f"({len(_out2):,} rows), network {'blocked throughout' if FORBID_NETWORK else 'not required'}")

### Reading the test result

`byte-identical` plus, on a blocked run, `network blocked throughout` — the socket block is
re-verified *after* all the work, so it cannot have been lifted midway and quietly restored.

What it does **not** prove: that a *fresh fetch* would reproduce these bytes. Covers could revise a
historical page, which is exactly why the raw HTML and its sha256 are pinned in the cache and the
manifest. Reproducibility here means "from these pinned inputs", which is the only kind an external
source can offer.

## Conclusion and next steps

**What was acquired.** 32 team rows for every season the archive publishes, parsed from pinned raw
HTML, mapped to canonical franchises, typed, and written to `futures/data/win_totals.csv` with full
provenance. The raw cache plus its manifest make the extract reproducible byte-for-byte with the
network disabled.

**What the source does not provide, and what it cost.**

1. **No sportsbook identity.** The page carries a number and both prices but never names the book.
   `book` is null on every row, so **G3-C fails on all 352** - and G3-C is the only key to §7 gate C.
   Writing "Covers" there would be inventing provenance to pass a gate.
2. **Date-granularity timing, mostly on kickoff day.** Only **2014-2018** carry an As-of date strictly
   before Week 1. Six more are dated *on* the Week 1 date, and 2012/2013/2023 carry no date at all
   (2023 says only *"Closing odds prior to each teams' first game"*).

**Verdict: `GO-TIER-B`** - `PREREGISTRATION.md` §10 **Amendment 1**, accepted by Joseph 2026-08-03
and frozen before notebooks `02`-`05` were opened. G3 was split: **G3-B** (point-in-time + named
`market_source`, `book` may be null) admits this data to §7 gates **A and B**; **G3-C** (named book,
strictly pre-kickoff) is unchanged, still fails, and still guards gate C. The kickoff-day clause
(A1.3) admits the same-day seasons on the source's own closing statement and records that **exact
closing timestamps are unavailable**; the strictly-dated seasons are carried as the mandatory A1.4
sensitivity (4 folds, underpowered, never the headline).

**Claim license.** Projection quality, and accuracy against an **archived market consensus of
unattributed sportsbook origin**, in aggregate. Not "the sportsbook line", not "Vegas", not "the
market". No sides, no probability against a posted line, no confidence tiers, no EV, no
profitability, and none of the words *bet*, *edge*, *lock*, *value*, *play*.

**Next steps.**

1. **`futures/season_team_totals/01_build_dataset.ipynb`** may now run - it reads the frozen fold
   sets from `data_audit.json` rather than choosing folds itself.
2. **`02`/`03`** report every number twice: headline (10 folds) and A1.4 strict subset (4 folds).
3. **`04`/`05`** only if §7 gate A passes; `05` may write no priced column.
4. **For Over/Under recommendations, collect 2026 lines prospectively** - timestamped, named-book.
   That is the only route to G3-C and therefore to gate C.

Full source description, settlement assumptions and the manual-download path:
`futures/DATA_SOURCE_NOTES.md`. Proposal of record (accepted, superseded by §10):
`futures/PROPOSED_AMENDMENT_FREE_MARKET_ARCHIVE.md`.
